# SSL Vision Foundation: one-click, auto-resuming Colab runner

**Every time: `Runtime → Run all`.** If Colab disconnects or times out, reconnect and *Run all* again;
it picks up exactly where it stopped.

* Everything that matters lives on **Google Drive** in `MyDrive/ssl-vision-foundation/`:
  `data/*.tar` (datasets, downloaded once), `checkpoints/` (saved every epoch), `results/`.
* Finished experiments are skipped; an interrupted one resumes from its last saved epoch.
* First run only: `Runtime → Change runtime type → T4 GPU`, and allow Drive access when asked.

Secrets (key icon in the left sidebar, switch *Notebook access* on):

| secret | why |
|---|---|
| `GH_TOKEN` | **needed while the GitHub repo is private** (fine-grained token, *Contents: read and write* on this repo). Also lets the last cell push results + README. Not needed once the repo is public (then only for the push). |
| `HF_TOKEN` | optional extra backup of checkpoints to the Hugging Face Hub (Drive is already enough). |

In [ ]:
# 1) Drive, secrets, code, dependencies
import os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/ssl-vision-foundation')
for d in ('data', 'checkpoints', 'results', 'exports', 'logs'):
    (DRIVE / d).mkdir(parents=True, exist_ok=True)

def secret(name):
    try:
        return userdata.get(name) or ''
    except Exception:
        return ''
GH_TOKEN, HF_TOKEN = secret('GH_TOKEN'), secret('HF_TOKEN')
if HF_TOKEN:
    os.environ.update(HF_TOKEN=HF_TOKEN, HF_REPO='ahmedashrafk12/ssl-vision-foundation', HF_PUSH='1')

REPO = Path('/content/ssl-vision-foundation')
URL = (f'https://{GH_TOKEN}@github.com/ahmedashrafk13/ssl-vision-foundation.git' if GH_TOKEN
       else 'https://github.com/ahmedashrafk13/ssl-vision-foundation.git')
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '-q', '--ff-only'], check=False)
else:
    r = subprocess.run(['git', 'clone', '-q', URL, str(REPO)], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit('git clone failed: the repo is private. Add a GH_TOKEN secret (see top) or make the repo public.')
os.chdir(REPO)

# checkpoints / results / exports / logs are symlinks into Drive, so every save survives a disconnect
for d in ('checkpoints', 'results', 'exports', 'logs'):
    p = REPO / d
    if p.is_symlink():
        continue
    if p.exists():
        subprocess.run(['cp', '-rn', f'{p}/.', str(DRIVE / d)])
        shutil.rmtree(p)
    p.symlink_to(DRIVE / d)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2) Data: downloaded once and stored on Drive as .tar; later sessions just unpack to fast local disk
LOCAL = REPO / 'data'
LOCAL.mkdir(exist_ok=True)

def fetch_neu():
    d = LOCAL / 'neu-det'
    d.mkdir(exist_ok=True)
    subprocess.run(['curl', '-sfL', '-o', str(d / 'neu.zip'),
                    'https://www.kaggle.com/api/v1/datasets/download/kaustubhdikshit/neu-surface-defect-database'], check=True)
    subprocess.run(['unzip', '-q', '-o', str(d / 'neu.zip'), '-d', str(d)], check=True)
    (d / 'neu.zip').unlink()

def fetch_cifar():
    import torchvision
    for train in (True, False):
        torchvision.datasets.CIFAR10(str(LOCAL / 'cifar'), train=train, download=True)

SOURCES = {
    'mvtec':       lambda: subprocess.run([sys.executable, 'scripts/download_data.py', 'mvtec'], check=True),
    'imagenet100': lambda: subprocess.run([sys.executable, 'scripts/download_data.py', 'imagenet100', '--per-class', '300'], check=True),
    'neu-det':     fetch_neu,
    'cifar':       fetch_cifar,
}
for name, fetch in SOURCES.items():
    tar = DRIVE / 'data' / f'{name}.tar'
    if (LOCAL / name / '.ready').exists():
        print(f'{name}: ready')
        continue
    if tar.exists():
        print(f'{name}: unpacking from Drive ...')
        subprocess.run(['tar', '-xf', str(tar), '-C', str(LOCAL)], check=True)
    else:
        print(f'{name}: first-time download (saved to Drive afterwards) ...')
        shutil.rmtree(LOCAL / name, ignore_errors=True)
        fetch()
        part = Path(str(tar) + '.part')
        subprocess.run(['tar', '-cf', str(part), '-C', str(LOCAL), name], check=True)
        part.rename(tar)  # only a complete archive ever gets the final name
    (LOCAL / name / '.ready').touch()
subprocess.run(['du', '-sh', *[str(LOCAL / n) for n in SOURCES]])

In [ ]:
# 3) Smoke test: every stage for a few steps (~5 min). Runs once; remembered on Drive.
marker = DRIVE / 'results' / '.smoke_ok'
if not marker.exists():
    subprocess.run([sys.executable, '-u', 'scripts/run_grid.py', '--smoke'], check=True)
    for d in (REPO / 'checkpoints').glob('*_smoke'):
        shutil.rmtree(d, ignore_errors=True)
    for f in (REPO / 'checkpoints' / 'finetune').glob('*_smoke*'):
        f.unlink()
    marker.touch()
else:
    print('smoke test already passed')

In [ ]:
# 4) The full experiment grid. Resumes automatically: finished runs are skipped, interrupted ones continue.
#    ~25-30 T4 hours in total, so expect several sessions. If Colab cuts out: reconnect, Run all.
if subprocess.run([sys.executable, '-u', 'scripts/run_grid.py']).returncode:
    raise SystemExit('Some stages failed twice; the FAILED summary just above shows why. Full logs: '
                     'Drive -> ssl-vision-foundation/logs/grid/. Everything else ran; Run all again to retry.')

In [ ]:
# 5) Results
from IPython.display import Markdown, Image, display
display(Markdown((REPO / 'results' / 'summary.md').read_text()))
for png in sorted((REPO / 'results').glob('*.png')):
    display(Image(str(png), width=520))

In [ ]:
# 6) Publish: write results into README.md and push to GitHub (needs GH_TOKEN with write access)
if GH_TOKEN:
    subprocess.run([sys.executable, 'scripts/update_readme.py'], check=True)
    pub = Path('/tmp/pub')
    shutil.rmtree(pub, ignore_errors=True)
    subprocess.run(['git', 'clone', '-q', URL, str(pub)], check=True)
    shutil.copytree(REPO / 'results', pub / 'results', dirs_exist_ok=True)
    shutil.copy(REPO / 'README.md', pub / 'README.md')
    g = lambda *a: subprocess.run(['git', '-C', str(pub), *a], check=True)
    g('add', 'README.md', 'results')
    if subprocess.run(['git', '-C', str(pub), 'diff', '--cached', '--quiet']).returncode:
        g('-c', 'user.name=Ahmed Ashraf', '-c', 'user.email=ahmedashrafk13@users.noreply.github.com',
          'commit', '-qm', 'Add experiment results')
        g('push', '-q')
        print('pushed results + README to GitHub')
    else:
        print('nothing new to push')
else:
    print('No GH_TOKEN: results are in Drive -> ssl-vision-foundation/results/')